In [26]:
import numpy as np


In [33]:
#define data

data = [
    [12.0, 1.5, 1, 'Wine'],
    [5.0, 2.0, 0, 'Beer'],
    [40.0, 0.0, 1, 'Whiskey'],
    [13.5, 1.2, 1, 'Wine'],
    [4.5, 1.8, 0, 'Beer'],
    [38.0, 0.1, 1, 'Whiskey'],
    [11.5, 1.7, 1, 'Wine'],
    [5.5, 2.3, 0, 'Beer']
]
#1st column is alcohol content, 2nd is sugar (g/l),3rd is color(0 is light and 1 is dark)
#convert to numpy array
data = np.array(data)

#split data into features(X) and labels(y)

label_map   = {'Wine': 0, 'Beer': 1, 'Whiskey': 2}
reverse_map = {v: k for k, v in label_map.items()}

X = np.array([[row[0], row[1], row[2]] for row in data], dtype=float)
y = np.array([label_map[row[3]] for row in data], dtype=int)

print(X)
print(y)

[[12.   1.5  1. ]
 [ 5.   2.   0. ]
 [40.   0.   1. ]
 [13.5  1.2  1. ]
 [ 4.5  1.8  0. ]
 [38.   0.1  1. ]
 [11.5  1.7  1. ]
 [ 5.5  2.3  0. ]]
[0 1 2 0 1 2 0 1]


In [34]:
def gini_impurity(labels):
    counts=np.unique(labels,return_counts=True)
    counts=np.array(counts[1])
    probs=counts/len(labels)
    return 1-np.sum(probs**2)

In [35]:
print(gini_impurity(y))

0.65625


In [36]:
# best split for a given feature is the one that minimizes gini impurity

def best_split(X, y):
    best_gini      = float('inf')
    best_feature   = None
    best_threshold = None

    for feature_idx in range(X.shape[1]):
        for threshold in np.unique(X[:, feature_idx]):
            left_mask  = X[:, feature_idx] <= threshold
            right_mask = ~left_mask

            y_left, y_right = y[left_mask], y[right_mask]

            if len(y_left) == 0 or len(y_right) == 0:
                continue

            weighted_gini = (len(y_left)  / len(y)) * gini_impurity(y_left) + \
                            (len(y_right) / len(y)) * gini_impurity(y_right)

            if weighted_gini < best_gini:
                best_gini      = weighted_gini
                best_feature   = feature_idx
                best_threshold = threshold

    return best_feature, best_threshold

In [37]:
#node and decision tree


class Node:
    def __init__(self, feature_index=None, threshold=None,
                 left=None, right=None, value=None):
        self.feature_index = feature_index
        self.threshold     = threshold
        self.left          = left
        self.right         = right
        self.value         = value          # set only for leaf nodes

    def is_leaf(self):
        return self.value is not None


class DecisionTree:
    def __init__(self, max_depth=5, min_samples=2):
        self.max_depth   = max_depth
        self.min_samples = min_samples
        self.root        = None             # set after fit()

    # ── Tree Building ────────────────────────────────────────────────

    def fit(self, X, y):
        """Public entry point — builds the tree and stores the root."""
        self.root = self._build_tree(X, y, depth=0)

    def _build_tree(self, X, y, depth):
        """Recursively build the tree and return the root Node."""
        # Stopping condition 1: pure node
        if len(np.unique(y)) == 1:
            return Node(value=y[0])

        # Stopping condition 2: max depth or too few samples
        if depth >= self.max_depth or len(y) < self.min_samples:
            return Node(value=self._majority_class(y))

        # Find best split using the standalone function
        feature_idx, threshold = best_split(X, y)

        # Stopping condition 3: no valid split found
        if feature_idx is None:
            return Node(value=self._majority_class(y))

        # Partition and recurse
        left_mask  = X[:, feature_idx] <= threshold
        right_mask = ~left_mask

        left_child  = self._build_tree(X[left_mask],  y[left_mask],  depth + 1)
        right_child = self._build_tree(X[right_mask], y[right_mask], depth + 1)

        return Node(feature_index=feature_idx,
                    threshold=threshold,
                    left=left_child,
                    right=right_child)
    #predicting 

    def predict(self, X):
        """Predict class labels for every row in X."""
        return np.array([self._predict_one(self.root, x) for x in X])

    def _predict_one(self, node, x):
        """Traverse the tree for a single sample x."""
        if node.is_leaf():
            return node.value
        if x[node.feature_index] <= node.threshold:
            return self._predict_one(node.left, x)
        else:
            return self._predict_one(node.right, x)

    

    def score(self, X, y):
        """Return accuracy as a percentage."""
        return np.mean(self.predict(X) == y) * 100

        

    def _majority_class(self, y):
        """Return the most frequent class label in y."""
        return np.bincount(y).argmax()

In [39]:
test_data = np.array([
    [6.0, 2.1, 0],   # Expected: Beer
    [39.0, 0.05, 1], # Expected: Whiskey
    [13.0, 1.3, 1]   # Expected: Wine
])


In [44]:
clf = DecisionTree(max_depth=5, min_samples=2)
clf.fit(X, y)
y_test = np.array([1,2,0])  # Beer=1, Whiskey=2, Wine=0
y_pred   = clf.predict(test_data)
accuracy = clf.score(test_data, y_test)

print(f"Predictions : {[reverse_map[p] for p in y_pred]}")
print(f"Ground Truth: {[reverse_map[t] for t in y_test]}")
print(f"Accuracy    : {accuracy:.1f}%")

Predictions : ['Wine', 'Whiskey', 'Wine']
Ground Truth: ['Beer', 'Whiskey', 'Wine']
Accuracy    : 66.7%
